In [30]:
# 青いブロックを箱へ入れてください
# Colabセル①
!pip -q install google-generativeai imageio imageio-ffmpeg numpy matplotlib

In [31]:
#Colabセル② Gemini設定
import google.generativeai as genai
import json

API_KEY = "AQ.Ab8RN6JfRt-KOPp4Mtk6QT9wsFvSWdUv7mH023VUgl3z-aKsqA"

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel(
    "gemini-3.5-flash"
)

In [32]:
# Colabセル③ 命令解析
def parse_command(command):

    prompt = f"""
    次の命令から対象色をJSONで返してください。

    命令:
    {command}

    出力例:
    {{
      "color":"blue"
    }}
    """

    response = model.generate_content(prompt)

    text = response.text
    text = text.replace("```json", "")
    text = text.replace("```", "")

    return json.loads(text)

In [33]:
# Colabセル④ シミュレーション定義
import numpy as np
import matplotlib.pyplot as plt
import imageio
import os

frames = []

BLOCKS = {
    "red":[2,3],
    "blue":[5,3],
    "green":[8,3]
}

BOX = [8,8]

In [34]:
# Colabセル⑤ 描画関数
def draw_scene(
    arm_x,
    arm_y,
    block_positions,
    title=""
):

    fig, ax = plt.subplots(figsize=(6,6))

    ax.set_xlim(0,10)
    ax.set_ylim(0,10)

    ax.set_title(title)

    ax.add_patch(
        plt.Rectangle(
            (7.3,7.3),
            1.4,
            1.4,
            fill=False,
            linewidth=3
        )
    )

    for color,pos in block_positions.items():

        ax.scatter(
            pos[0],
            pos[1],
            s=1000,
            c=color
        )

    ax.plot(
        [1,arm_x],
        [1,arm_y],
        linewidth=5,
        c="black"
    )

    ax.scatter(
        arm_x,
        arm_y,
        s=300,
        c="black"
    )

    fig.canvas.draw()

    img = np.array(
        fig.canvas.buffer_rgba()
    )

    plt.close()

    return img

In [35]:
#Colabセル⑥ アニメーション生成
def move_linear(
    start,
    end,
    steps
):

    xs = np.linspace(
        start[0],
        end[0],
        steps
    )

    ys = np.linspace(
        start[1],
        end[1],
        steps
    )

    return list(zip(xs,ys))

In [36]:
# Colabセル⑦ ロボットタスク
def run_task(target_color):

    global frames

    block_positions = {
        "red":[2,3],
        "blue":[5,3],
        "green":[8,3]
    }

    target_pos = block_positions[target_color]

    arm_home = [1,1]

    path1 = move_linear(
        arm_home,
        target_pos,
        50
    )

    for p in path1:

        frames.append(
            draw_scene(
                p[0],
                p[1],
                block_positions,
                f"Move to {target_color}"
            )
        )

    path2 = move_linear(
        target_pos,
        BOX,
        70
    )

    for p in path2:

        block_positions[target_color] = [
            p[0],
            p[1]
        ]

        frames.append(
            draw_scene(
                p[0],
                p[1],
                block_positions,
                f"Transport {target_color}"
            )
        )

    for _ in range(30):

        frames.append(
            draw_scene(
                BOX[0],
                BOX[1],
                block_positions,
                "Success"
            )
        )

In [37]:
#Colabセル⑧ AI指示
commands = [
    "青いブロックを箱へ入れてください",
    "赤いブロックを箱へ入れてください",
    "緑のブロックを箱へ入れてください"
]

In [38]:
# Colabセル⑨ 実行
for command in commands:

    result = parse_command(
        command
    )

    color = result["color"]

    print(
        command,
        "->",
        color
    )

    run_task(color)

青いブロックを箱へ入れてください -> blue
赤いブロックを箱へ入れてください -> red
緑のブロックを箱へ入れてください -> green


In [39]:
# Colabセル⑩ 90秒動画出力
imageio.mimsave(
    "PAI_Final_Task.mp4",
    frames,
    fps=4
)

print("saved")

saved


In [40]:
# Colabセル⑪ ダウンロード
from google.colab import files

files.download(
    "PAI_Final_Task.mp4"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>